In [2]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [13]:
data = pd.read_csv("../data/adult.csv", index_col=[0])

data = data.replace("?", np.nan)
data = data.dropna()

expected_salary = np.where(data["salary K$"] > 50, ">50K", "<=50K")
mismatch_mask = data["salary"] != expected_salary

data[mismatch_mask]

,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,salary,salary K$


# Task 1
Print the count of men and women in the dataset.

In [14]:
data["sex"].value_counts()

sex
Male      20380
Female     9782
Name: count, dtype: int64

# Task 2
Find the average age of men in dataset

In [21]:
data[data["sex"] == "Male"]["age"].mean()

np.float64(39.18400392541707)

# Task 3
Get the percentage of people from Poland (native-country)

In [24]:
(data["native-country"] == "Poland").sum() / len(data) * 100

np.float64(0.18566408063125786)

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [26]:
data.groupby("salary")["age"].agg(["mean", "std"])

,mean,std
salary,,
<=50K,36.60806,13.464631
>50K,43.95911,10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [28]:
higher_education = ["Bachelors", "Prof-school", "Assoc-acdm", "Assoc-voc", "Masters", "Doctorate"]
no_higher_edu = ~data["education"].isin(higher_education)
high_salary = data["salary"] == ">50K"

mask = no_higher_edu & high_salary
data[mask]

,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,salary,salary K$
7,52,Self-emp-not-inc,HS-grad,Married-civ-spouse,Exec-managerial,Husband,White,Male,45,United-States,>50K,307
10,37,Private,Some-college,Married-civ-spouse,Exec-managerial,Husband,Black,Male,80,United-States,>50K,116
55,43,Private,Some-college,Married-civ-spouse,Tech-support,Husband,White,Male,40,United-States,>50K,341
67,53,Private,HS-grad,Married-civ-spouse,Adm-clerical,Wife,White,Female,40,United-States,>50K,225
68,49,Self-emp-inc,Some-college,Married-civ-spouse,Exec-managerial,Husband,White,Male,50,United-States,>50K,194
...,...,...,...,...,...,...,...,...,...,...,...,...
32462,48,Self-emp-inc,HS-grad,Married-civ-spouse,Craft-repair,Husband,White,Male,50,United-States,>50K,343
32518,57,Local-gov,HS-grad,Married-civ-spouse,Craft-repair,Husband,White,Male,40,United-States,>50K,116
32519,46,Private,Some-college,Married-civ-spouse,Exec-managerial,Husband,White,Male,48,United-States,>50K,239
32557,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,40,United-States,>50K,173


# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [30]:
data.groupby("education")["age"].describe()

,count,mean,std,min,25%,50%,75%,max
education,,,,,,,,
10th,820.0,37.897561,16.225795,17.0,23.0,36.0,52.0,90.0
11th,1048.0,32.363550,15.089307,17.0,18.0,28.5,43.0,90.0
12th,377.0,32.013263,14.373710,17.0,19.0,28.0,41.0,79.0
1st-4th,151.0,44.622517,14.929051,19.0,33.0,44.0,56.0,81.0
5th-6th,288.0,41.649306,14.754622,17.0,28.0,41.0,53.0,82.0
7th-8th,557.0,47.631957,15.737479,17.0,34.0,49.0,60.0,90.0
9th,455.0,40.303297,15.335754,17.0,28.0,38.0,53.0,90.0
Assoc-acdm,1008.0,37.286706,10.509755,19.0,29.0,36.0,44.0,90.0
Assoc-voc,1307.0,38.246366,11.181253,19.0,30.0,37.0,45.0,84.0


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [31]:
men = data[data["sex"] == "Male"]
is_married = men["marital-status"].str.startswith("Married")
men.groupby(is_married)["salary"].value_counts(normalize=True)

marital-status  salary
False           <=50K     0.911506
                >50K      0.088494
True            <=50K     0.552016
                >50K      0.447984
Name: proportion, dtype: float64

# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [33]:
max_hours = data["hours-per-week"].max()
people_count = (data["hours-per-week"] == max_hours).sum()
people_count

np.int64(78)

# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [35]:
data_encoded = pd.get_dummies(data)

full_corr = data_encoded.corr()

mask = np.triu(np.ones(full_corr.shape), k=1).astype(bool)
corr_pairs = full_corr.where(mask).stack()

strongest = corr_pairs.abs().sort_values(ascending=False)

strongest.head(20)

salary_<=50K                       salary_>50K                     1.000000
sex_Female                         sex_Male                        1.000000
marital-status_Married-civ-spouse  relationship_Husband            0.896502
salary K$                          salary_>50K                     0.853894
                                   salary_<=50K                    0.853894
race_Black                         race_White                      0.794808
marital-status_Married-civ-spouse  marital-status_Never-married    0.644862
relationship_Husband               sex_Male                        0.581221
                                   sex_Female                      0.581221
marital-status_Never-married       relationship_Husband            0.578903
marital-status_Married-civ-spouse  relationship_Not-in-family      0.546401
age                                marital-status_Never-married    0.524003
workclass_Private                  workclass_Self-emp-not-inc      0.505588
marital-stat